# 6. Two-Optimizer Adversarial cVAE + MMD Batch Correction

**Pipeline:**
1. Load cohort from `data/interim_PT/`
2. Generate dummy split (`Split_dummy`: 60/20/20)
3. Fit Adv2-cVAE on **train** embeddings only
4. Transform both train and test -> 128-dim batch-invariant latent
5. Save to `data/processed_two_opt/`
6. Compare with KL-cVAE results

## 0. Environment Setup

In [ ]:
# !rm -rf batchcor-rna-embeds/  # if already cloned
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY,
    SCVI_CORRECTED_KEY, SCVI_LATENT_DIM,
    COHORT_CANCER_CODE,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.split_utils import add_dummy_split, get_split_masks
from batchcor_rna_emb.batch_correction.scvi_corrector import (
    CVAEAdv2Config, CVAEAdv2Corrector,
)
from batchcor_rna_emb.metrics.batch_metrics import (
    compute_kbet, compute_ilisi, compute_asw_batch, compute_graph_connectivity,
)
from batchcor_rna_emb.metrics.bio_metrics import (
    compute_clisi, compute_silhouette_bio, compute_nmi, compute_ari,
    run_leiden_clustering,
)
from batchcor_rna_emb.metrics.aggregation import geometric_mean

set_logger(level="INFO")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Imports OK, device: {}", DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')
DATA_PROCESSED_ADV  = Path('/content/drive/MyDrive/data/processed_two_opt')
DATA_PROCESSED_ADV.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR  = Path('/content/drive/MyDrive/data/processed')

SPLIT_COL = "Split_dummy"
ADV2_CORRECTED_KEY = COMPASS_PT_EMBEDDING_KEY + "_Adv2_corrected"
MULTI_BATCH_COHORTS = ["PUB_KIRC_ICI_combined", "NSCLC_Tissue_ICI_Pred", "PUB_BRCA_SCANB"]

logger.info("Paths configured")
logger.info("Source: {}", DATA_INTERIM_PT_DIR)
logger.info("Dest:   {}", DATA_PROCESSED_ADV)

## 2. Load, Correct & Save

In [ ]:
results = {}

for cohort_name in MULTI_BATCH_COHORTS:
    logger.info("\n" + "=" * 70)
    logger.info("Processing: {}", cohort_name)
    logger.info("=" * 70)

    src_path = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"
    try:
        adata = load_cohort(src_path)
    except Exception as e:
        logger.warning("Failed to load {}, skipping. Error: {}", cohort_name, e)
        continue
    logger.info("Loaded: {} samples x {} genes", adata.n_obs, adata.n_vars)

    add_dummy_split(adata, seed=SEED)
    train_mask, test_mask = get_split_masks(adata, SPLIT_COL)
    logger.info("Split: {} train, {} test, {} NaN",
                train_mask.sum(), test_mask.sum(),
                (~train_mask & ~test_mask).sum())

    X_all = adata.obsm[COMPASS_PT_EMBEDDING_KEY].astype(np.float32)
    batch_all = adata.obs[BATCH_COL].values

    n_batches = len(adata.obs[BATCH_COL].cat.categories)
    cfg = CVAEAdv2Config(
        latent_dim=SCVI_LATENT_DIM,
        n_epochs=300,
        normalize=True,
    )
    logger.info("Config: latent_dim={}, n_batches={}, n_epochs={}",
                cfg.latent_dim, n_batches, cfg.n_epochs)

    corrector = CVAEAdv2Corrector(cfg)
    corrector.fit(X_all[train_mask], batch_all[train_mask])
    history = corrector.history_

    Z_corrected = corrector.transform(X_all, batch_all)
    adata.obsm[ADV2_CORRECTED_KEY] = Z_corrected
    logger.info("Corrected embedding shape: {}", Z_corrected.shape)

    dst_path = DATA_PROCESSED_ADV / f"{cohort_name}.adata.zarr"
    save_adata_zarr(adata, dst_path)
    logger.info("Saved to: {}", dst_path)

    results[cohort_name] = {"adata": adata, "history": history}

logger.info("\nAll cohorts processed!")

## 8. Immotion ICI + TKI Combined (Over-correction Control)

Tests if the adversarial network hallucinates batch corrections when there is no technical batch effect (only treatment arms from the same lab).

In [ ]:
IMMOTION_ICI = "PUB_ccRCC_Immotion150_and_151_ICI"
IMMOTION_TKI = "PUB_ccRCC_Immotion150_and_151_TKI"
IMMOTION_COMBINED = "PUB_ccRCC_Immotion_combined"

logger.info("\n" + "=" * 70)
logger.info("Processing Over-correction Control: {}", IMMOTION_COMBINED)

ad_ici = load_cohort(DATA_INTERIM_PT_DIR / f"{IMMOTION_ICI}.adata.zarr")
ad_tki = load_cohort(DATA_INTERIM_PT_DIR / f"{IMMOTION_TKI}.adata.zarr")
ad_ici.obs["Immotion_arm"] = "ICI"
ad_tki.obs["Immotion_arm"] = "TKI"

import anndata as ad
adata_imm = ad.concat([ad_ici, ad_tki], merge="same")
adata_imm.obs["Immotion_arm"] = adata_imm.obs["Immotion_arm"].astype("category")

add_dummy_split(adata_imm, seed=SEED)
train_mask, _ = get_split_masks(adata_imm, SPLIT_COL)

X_imm = adata_imm.obsm[COMPASS_PT_EMBEDDING_KEY].astype(np.float32)
batch_imm = adata_imm.obs["Immotion_arm"].values

cfg_imm = CVAEAdv2Config(latent_dim=SCVI_LATENT_DIM, n_epochs=300, normalize=True)
corrector_imm = CVAEAdv2Corrector(cfg_imm)
corrector_imm.fit(X_imm[train_mask], batch_imm[train_mask])

adata_imm.obsm[ADV2_CORRECTED_KEY] = corrector_imm.transform(X_imm, batch_imm)

dst_imm = DATA_PROCESSED_ADV / f"{IMMOTION_COMBINED}.adata.zarr"
save_adata_zarr(adata_imm, dst_imm)
logger.info("Saved Immotion Combined to: {}", dst_imm)

results[IMMOTION_COMBINED] = {"adata": adata_imm, "history": corrector_imm.history_}


## 9. Mariathasan Permutation Test

Assigns random fake batch labels to a single-batch cohort to ensure the network doesn't arbitrarily split data.

In [ ]:
BLCA_NAME = "PUB_BLCA_Mariathasan_EGAS00001002556"

logger.info("\n" + "=" * 70)
logger.info("Processing Permutation Control: {}", BLCA_NAME)

adata_blca = load_cohort(DATA_INTERIM_PT_DIR / f"{BLCA_NAME}.adata.zarr")

# Assign random fake batches
np.random.seed(SEED)
fake_batches = np.random.choice(["Fake1", "Fake2", "Fake3"], size=adata_blca.n_obs)
adata_blca.obs["Fake_Batch"] = pd.Categorical(fake_batches)

add_dummy_split(adata_blca, seed=SEED)
train_mask, _ = get_split_masks(adata_blca, SPLIT_COL)

X_blca = adata_blca.obsm[COMPASS_PT_EMBEDDING_KEY].astype(np.float32)
batch_blca = adata_blca.obs["Fake_Batch"].values

cfg_blca = CVAEAdv2Config(latent_dim=SCVI_LATENT_DIM, n_epochs=300, normalize=True)
corrector_blca = CVAEAdv2Corrector(cfg_blca)
corrector_blca.fit(X_blca[train_mask], batch_blca[train_mask])

adata_blca.obsm[ADV2_CORRECTED_KEY] = corrector_blca.transform(X_blca, batch_blca)

dst_blca = DATA_PROCESSED_ADV / f"{BLCA_NAME}.adata.zarr"
save_adata_zarr(adata_blca, dst_blca)
logger.info("Saved Mariathasan Permutation to: {}", dst_blca)

results[BLCA_NAME] = {"adata": adata_blca, "history": corrector_blca.history_}


## 3. Training Curves

In [ ]:
for cohort_name, res in results.items():
    history = res["history"]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle(f"{cohort_name} — Training Curves", fontsize=14, fontweight="bold")

    axes[0].plot(history.recon, label="Recon (MSE)")
    axes[0].set_title("Reconstruction Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")

    if hasattr(history, "disc_accuracy") and history.disc_accuracy:
        axes[1].plot(history.disc_accuracy, label="Disc Accuracy", color="orange")
        axes[1].axhline(1.0/res["adata"].obs[BATCH_COL].nunique(), ls="--", color="gray", label="Chance")
        axes[1].set_title("Discriminator Accuracy")
        axes[1].set_xlabel("Epoch"); axes[1].legend()

    if hasattr(history, "adv_enc") and history.adv_enc:
        axes[2].plot(history.adv_enc, label="Adversarial Loss", color="red")
        axes[2].set_title("Adversarial Loss (encoder)")
        axes[2].set_xlabel("Epoch")

    if hasattr(history, "mmd") and history.mmd:
        axes[3].plot(history.mmd, label="MMD Loss", color="green")
        axes[3].set_title("MMD Loss")
        axes[3].set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

## 4. UMAP Before vs After

In [ ]:
from umap import UMAP

for cohort_name, res in results.items():
    adata = res["adata"]
    X_before = adata.obsm[COMPASS_PT_EMBEDDING_KEY]
    X_after  = adata.obsm[ADV2_CORRECTED_KEY]

    umap_before = UMAP(n_components=2, random_state=SEED).fit_transform(X_before)
    umap_after  = UMAP(n_components=2, random_state=SEED).fit_transform(X_after)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"{cohort_name} — UMAP Before vs After Adv2-cVAE", fontsize=14, fontweight="bold")

    bcol = "Immotion_arm" if cohort_name == "PUB_ccRCC_Immotion_combined" else ("Fake_Batch" if cohort_name == "PUB_BLCA_Mariathasan_EGAS00001002556" else BATCH_COL)
    batch_labels = adata.obs[bcol].values
    diag_labels  = adata.obs[DIAGNOSIS_COL].values if DIAGNOSIS_COL in adata.obs.columns else None

    for j, (emb, title) in enumerate([(umap_before, "Before"), (umap_after, "After Adv2")]):
        ax = axes[0, j]
        for b in sorted(set(batch_labels)):
            mask = batch_labels == b
            ax.scatter(emb[mask, 0], emb[mask, 1], s=8, alpha=0.5, label=b)
        ax.set_title(f"{title} — RNA Batch")
        ax.legend(fontsize=6, markerscale=2, loc="best")

        if diag_labels is not None:
            ax = axes[1, j]
            for d in sorted(set(diag_labels)):
                mask = diag_labels == d
                ax.scatter(emb[mask, 0], emb[mask, 1], s=8, alpha=0.5, label=d)
            ax.set_title(f"{title} — Diagnosis")
            ax.legend(fontsize=6, markerscale=2, loc="best")

    plt.tight_layout()
    plt.show()

## 5. Silhouette Score Summary

In [ ]:
from sklearn.metrics import silhouette_score

rows = []
for cohort_name, res in results.items():
    adata = res["adata"]
    bcol = "Immotion_arm" if cohort_name == "PUB_ccRCC_Immotion_combined" else ("Fake_Batch" if cohort_name == "PUB_BLCA_Mariathasan_EGAS00001002556" else BATCH_COL)
    batch = adata.obs[bcol].values
    diag  = adata.obs[DIAGNOSIS_COL].values if DIAGNOSIS_COL in adata.obs.columns else None

    for key, label in [(COMPASS_PT_EMBEDDING_KEY, "Raw PT"),
                       (ADV2_CORRECTED_KEY, "Adv2-cVAE")]:
        emb = adata.obsm[key]
        sil_b = silhouette_score(emb, batch) if len(set(batch)) > 1 else np.nan
        sil_d = silhouette_score(emb, diag) if diag is not None and len(set(diag)) > 1 else np.nan
        rows.append({"Cohort": cohort_name, "Method": label,
                     "Sil_Batch": round(sil_b, 4) if not np.isnan(sil_b) else np.nan,
                     "Sil_Diag": round(sil_d, 4) if not np.isnan(sil_d) else np.nan})

df_sil = pd.DataFrame(rows)
display(df_sil)

## 6. Comparison with KL-cVAE (Notebook 4)

In [ ]:
rows_cmp = []

for cohort_name in results.keys():
    adv_adata = results[cohort_name]["adata"]
    bcol = "Immotion_arm" if cohort_name == "PUB_ccRCC_Immotion_combined" else ("Fake_Batch" if cohort_name == "PUB_BLCA_Mariathasan_EGAS00001002556" else BATCH_COL)
    batch = adv_adata.obs[bcol].values
    n_uniq = len(set(batch))

    emb_adv = adv_adata.obsm[ADV2_CORRECTED_KEY]
    sil_adv = silhouette_score(emb_adv, batch) if n_uniq > 1 else np.nan

    kl_path = DATA_PROCESSED_DIR / f"{cohort_name}.adata.zarr"
    sil_kl = np.nan
    if kl_path.exists():
        try:
            kl_adata = load_cohort(kl_path)
            if SCVI_CORRECTED_KEY in kl_adata.obsm:
                emb_kl = kl_adata.obsm[SCVI_CORRECTED_KEY]
                sil_kl = silhouette_score(emb_kl, kl_adata.obs[bcol].values if bcol in kl_adata.obs else batch) if n_uniq > 1 else np.nan
        except Exception as e:
            logger.warning("Could not load KL-cVAE result for {}: {}", cohort_name, e)

    emb_raw = adv_adata.obsm[COMPASS_PT_EMBEDDING_KEY]
    sil_raw = silhouette_score(emb_raw, batch) if n_uniq > 1 else np.nan

    rows_cmp.append({
        "Cohort": cohort_name,
        "Sil_Batch_Raw": round(sil_raw, 4),
        "Sil_Batch_KL_cVAE": round(sil_kl, 4) if not np.isnan(sil_kl) else np.nan,
        "Sil_Batch_Adv2": round(sil_adv, 4),
    })

df_cmp = pd.DataFrame(rows_cmp)
display(df_cmp)

## 7. Full scIB Metrics (KIRC + NSCLC)

In [ ]:
scib_rows = []

for cohort_name, res in results.items():
    adata = res["adata"]
    batch_col = "Immotion_arm" if cohort_name == "PUB_ccRCC_Immotion_combined" else ("Fake_Batch" if cohort_name == "PUB_BLCA_Mariathasan_EGAS00001002556" else BATCH_COL)
    diag_col  = DIAGNOSIS_COL if DIAGNOSIS_COL in adata.obs.columns else None

    for key, method in [(COMPASS_PT_EMBEDDING_KEY, "Raw_PT"),
                        (ADV2_CORRECTED_KEY, "Adv2_cVAE")]:
        row = {"Cohort": cohort_name, "Method": method}

        try: row["ASW_batch"] = round(compute_asw_batch(adata, key, batch_col), 4)
        except: row["ASW_batch"] = np.nan

        if diag_col and len(adata.obs[diag_col].unique()) > 1:
            try: row["Sil_bio"] = round(compute_silhouette_bio(adata, key, diag_col), 4)
            except: row["Sil_bio"] = np.nan
        else:
            row["Sil_bio"] = np.nan

        scib_rows.append(row)

df_scib = pd.DataFrame(scib_rows)
display(df_scib)


## 10. Visualization (PCA Scatter Plots)
Visualizing PC1 vs PC2 to inspect batch mixing before and after Adv2-cVAE correction.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

for cohort_name, res in results.items():
    adata = res["adata"]
    batch_col = "Immotion_arm" if cohort_name == "PUB_ccRCC_Immotion_combined" else ("Fake_Batch" if cohort_name == "PUB_BLCA_Mariathasan_EGAS00001002556" else BATCH_COL)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"{cohort_name}: PCA Visualization (colored by {batch_col})")
    
    for i, (key, title) in enumerate([(COMPASS_PT_EMBEDDING_KEY, "Raw PT Embeddings"), 
                                      (ADV2_CORRECTED_KEY, "Adv2-cVAE Corrected")]):
        if key not in adata.obsm:
            continue
            
        X = adata.obsm[key]
        pca = PCA(n_components=2, random_state=42)
        X_pca = pca.fit_transform(X)
        
        sns.scatterplot(
            x=X_pca[:, 0], y=X_pca[:, 1], 
            hue=adata.obs[batch_col].astype(str), 
            palette="Set1", s=15, alpha=0.8, ax=axes[i]
        )
        axes[i].set_title(title)
        axes[i].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
        axes[i].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
        axes[i].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()
